In [1]:
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, BRONZE_DB, SILVER_DB

# ETL Bronze → Silver
Transforma los datos de Bronze aplicando:
- `trimBoth()`: elimina espacios en blanco al inicio y final
- `ifNull(col, '')`: reemplaza NULL por string vacío en campos de texto
- `ifNull(col, 0)`: reemplaza NULL por 0 en campos numéricos
- `concat(first_name, ' ', last_name)`: genera campo `full_name`
- Filtro `WHERE rental_date IS NOT NULL`: elimina rentas sin fecha
- Conversión NaT → None para compatibilidad con ClickHouse

In [2]:
def cargar_a_silver(nombre_tabla, query):
    print(f'Leyendo bronze.raw_{nombre_tabla}...')
    rows, cols = CH.execute(query, with_column_types=True)
    df = pd.DataFrame(rows, columns=[c[0] for c in cols])

    # Convertir NaT → None en todas las columnas
    for col in df.columns:
        if df[col].dtype == 'object' or str(df[col].dtype).startswith('datetime'):
            try:
                mask = df[col].isna()
                df[col] = df[col].astype(object)
                df.loc[mask, col] = None
            except:
                pass

    df['_processed_at'] = datetime.now()
    df = df.where(pd.notnull(df), None)

    CH.execute(f'TRUNCATE TABLE {SILVER_DB}.stg_{nombre_tabla}')
    CH.execute(f'INSERT INTO {SILVER_DB}.stg_{nombre_tabla} VALUES', df.to_dict('records'))
    print(f'  ✓ {len(df):,} filas cargadas en silver.stg_{nombre_tabla}')

In [3]:
# Transformaciones SQL por tabla
tablas = {
    'actor': f"""
        SELECT
            actor_id,
            trimBoth(first_name) AS first_name,
            trimBoth(last_name)  AS last_name,
            concat(trimBoth(first_name), ' ', trimBoth(last_name)) AS full_name
        FROM {BRONZE_DB}.raw_actor
    """,
    'address': f"""
        SELECT
            address_id,
            trimBoth(address)                 AS address,
            trimBoth(ifNull(address2,  ''))   AS address2,
            trimBoth(district)                AS district,
            city_id,
            trimBoth(ifNull(postal_code, '')) AS postal_code,
            trimBoth(phone)                   AS phone
        FROM {BRONZE_DB}.raw_address
    """,
    'category': f"""
        SELECT category_id, trimBoth(name) AS name
        FROM {BRONZE_DB}.raw_category
    """,
    'city': f"""
        SELECT city_id, trimBoth(city) AS city, country_id
        FROM {BRONZE_DB}.raw_city
    """,
    'country': f"""
        SELECT country_id, trimBoth(country) AS country
        FROM {BRONZE_DB}.raw_country
    """,
    'customer': f"""
        SELECT
            customer_id,
            store_id,
            trimBoth(first_name) AS first_name,
            trimBoth(last_name)  AS last_name,
            concat(trimBoth(first_name), ' ', trimBoth(last_name)) AS full_name,
            trimBoth(ifNull(email, '')) AS email,
            address_id,
            active,
            create_date
        FROM {BRONZE_DB}.raw_customer
    """,
    'film': f"""
        SELECT
            film_id,
            trimBoth(title)                        AS title,
            trimBoth(ifNull(description,    ''))   AS description,
            ifNull(release_year, 0)                AS release_year,
            language_id,
            rental_duration,
            rental_rate,
            ifNull(length, 0)                      AS length,
            replacement_cost,
            trimBoth(ifNull(rating,          ''))  AS rating,
            trimBoth(ifNull(special_features,''))  AS special_features
        FROM {BRONZE_DB}.raw_film
    """,
    'film_actor': f"""
        SELECT actor_id, film_id
        FROM {BRONZE_DB}.raw_film_actor
    """,
    'film_category': f"""
        SELECT film_id, category_id
        FROM {BRONZE_DB}.raw_film_category
    """,
    'inventory': f"""
        SELECT inventory_id, film_id, store_id
        FROM {BRONZE_DB}.raw_inventory
    """,
    'language': f"""
        SELECT language_id, trimBoth(name) AS name
        FROM {BRONZE_DB}.raw_language
    """,
    'payment': f"""
        SELECT
            payment_id,
            customer_id,
            staff_id,
            ifNull(rental_id, 0) AS rental_id,
            amount,
            payment_date
        FROM {BRONZE_DB}.raw_payment
    """,
    'rental': f"""
        SELECT
            rental_id,
            rental_date,
            inventory_id,
            customer_id,
            return_date,
            staff_id
        FROM {BRONZE_DB}.raw_rental
        WHERE rental_date IS NOT NULL
    """,
    'staff': f"""
        SELECT
            staff_id,
            trimBoth(first_name) AS first_name,
            trimBoth(last_name)  AS last_name,
            concat(trimBoth(first_name), ' ', trimBoth(last_name)) AS full_name,
            trimBoth(ifNull(email,   '')) AS email,
            store_id,
            active,
            trimBoth(username) AS username
        FROM {BRONZE_DB}.raw_staff
    """,
    'store': f"""
        SELECT store_id, manager_staff_id, address_id
        FROM {BRONZE_DB}.raw_store
    """,
}

In [4]:
# Ejecutar transformación Bronze → Silver
print('=== ETL Bronze → Silver ===')
for tabla, query in tablas.items():
    cargar_a_silver(tabla, query)
print('\n=== Carga Silver completada ===')

=== ETL Bronze → Silver ===
Leyendo bronze.raw_actor...
  ✓ 200 filas cargadas en silver.stg_actor
Leyendo bronze.raw_address...
  ✓ 603 filas cargadas en silver.stg_address
Leyendo bronze.raw_category...
  ✓ 16 filas cargadas en silver.stg_category
Leyendo bronze.raw_city...
  ✓ 600 filas cargadas en silver.stg_city
Leyendo bronze.raw_country...
  ✓ 109 filas cargadas en silver.stg_country
Leyendo bronze.raw_customer...
  ✓ 599 filas cargadas en silver.stg_customer
Leyendo bronze.raw_film...
  ✓ 1,000 filas cargadas en silver.stg_film
Leyendo bronze.raw_film_actor...
  ✓ 5,462 filas cargadas en silver.stg_film_actor
Leyendo bronze.raw_film_category...
  ✓ 1,000 filas cargadas en silver.stg_film_category
Leyendo bronze.raw_inventory...
  ✓ 4,581 filas cargadas en silver.stg_inventory
Leyendo bronze.raw_language...
  ✓ 6 filas cargadas en silver.stg_language
Leyendo bronze.raw_payment...
  ✓ 16,044 filas cargadas en silver.stg_payment
Leyendo bronze.raw_rental...
  ✓ 16,044 filas cargad